In [ ]:
#| hide
from comm_core import *

# comm-core

> Useful abstractions for multi-agent learning with communication.

comm-core models the communication between agents that learn, whatever they learn: MARL agents, multi-agent world
models, federated learning clients.

* **The learning code** decides when to communicate, what, and to whom.
* **comm-core** decides whether the message can go, when it goes out, what arrives, and what it costs: losses, bit
  errors, interference, bits, latency, energy.

The same learning code then runs over a perfect channel, a lossy one or a ray-traced radio environment, by changing
how the network is built.

See [Design](https://Ahmed-Khaled-Saleh.github.io/comm-core/design.html) for how the pieces fit together.

## Install

```sh
pip install git+https://github.com/Ahmed-Khaled-Saleh/comm-core.git
```

Its only dependency is numpy. The vectorized link-level functions also take torch tensors, if torch is installed.

## How to use

Two agents share a latent over a radio link at 6 dB SNR, where packets get lost:

In [ ]:
agents = ['agent_0', 'agent_1']
net = Network(
    nodes=[Node(a) for a in agents],
    topology=FullMeshTopology(agents),                       # who can talk to whom
    protocol=IdealProtocol(),                                # when messages go out
    backend=AnalyticalBackend(SystemLevelChannel(noise_power=0.25, seed=0)),   # what happens to them
)
for step in range(100):
    net.send(Message('agent_0', 'agent_1', payload=[0.1, 0.2, 0.3], type='latent', size_bits=96))
    net.step()

received = net.receive('agent_1')
m = net.metrics.summary()
print(f'{len(received)} of 100 latents arrived ({m.packet_loss:.0%} lost), {m.bits_sent} bits sent')

73 of 100 latents arrived (27% lost), 9600 bits sent


To change what happens to the messages, swap the channel and leave the learning code as it is:

* `IdentityChannel()` is perfect.
* `SystemLevelChannel(...)` loses packets with the error rate of their SNR.
* `LinkLevelChannel(...)` simulates every bit, so payloads arrive with bit errors.

To decide when messages go out, swap the protocol, e.g. `TDMAProtocol`.

## Documentation

* [Design principles](https://Ahmed-Khaled-Saleh.github.io/comm-core/principles.html): the rules comm-core is built by, and why.
* [Architecture](https://Ahmed-Khaled-Saleh.github.io/comm-core/design.html): the pieces, the life of a message, time, fidelity, interference, extending.
* **Examples**, with their results:
  * [Agents sharing what they see](https://Ahmed-Khaled-Saleh.github.io/comm-core/examples/sharing.html): losses that depend on where agents are.
  * [Medium access and interference](https://Ahmed-Khaled-Saleh.github.io/comm-core/examples/medium_access.html): throughput and staleness as agents
    are added.
  * [Channels at continuous positions](https://Ahmed-Khaled-Saleh.github.io/comm-core/examples/continuous_positions.html): ray-traced paths moved to
    any position.
  * [Levels of fidelity](https://Ahmed-Khaled-Saleh.github.io/comm-core/examples/fidelity.html): from an analytic formula to every bit of 5G NR.
  * [Robot control over Wi-Fi](https://Ahmed-Khaled-Saleh.github.io/comm-core/examples/robot_control.html): a custom channel model, built from the
    pieces.
* **Core:** [core](https://Ahmed-Khaled-Saleh.github.io/comm-core/core.html), [topology](https://Ahmed-Khaled-Saleh.github.io/comm-core/topology.html), [channel](https://Ahmed-Khaled-Saleh.github.io/comm-core/channel.html),
  [protocol](https://Ahmed-Khaled-Saleh.github.io/comm-core/protocol.html), [backend](https://Ahmed-Khaled-Saleh.github.io/comm-core/backend.html), [network](https://Ahmed-Khaled-Saleh.github.io/comm-core/network.html),
  [metrics](https://Ahmed-Khaled-Saleh.github.io/comm-core/metrics.html).
* **Radio:** [propagation](https://Ahmed-Khaled-Saleh.github.io/comm-core/propagation.html), [system level](https://Ahmed-Khaled-Saleh.github.io/comm-core/system.html),
  [link level](https://Ahmed-Khaled-Saleh.github.io/comm-core/link.html), [Sionna RT](https://Ahmed-Khaled-Saleh.github.io/comm-core/sionna_rt.html), [Sionna SYS](https://Ahmed-Khaled-Saleh.github.io/comm-core/sionna_sys.html),
  [Sionna PHY](https://Ahmed-Khaled-Saleh.github.io/comm-core/sionna_phy.html).

## Developer guide

comm-core is written with [nbdev](https://nbdev.fast.ai): the notebooks in `nbs/` are the source, and
`comm_core/` is generated from them.

```sh
pip install -e .
# edit the notebooks in nbs/, then:
nbdev_prepare        # export the library, run the notebooks' tests, clean them, update the README
```

Use nbdev 3.3 or newer: CI checks that the library matches what its nbdev exports.